# GeoCPU v48：40,500からの不足分野追加学習

旧422,500行＋v47追加848行を減らさないことをCPU事前検査で確認してからGPU実行します。

非公開GitHubの読み込みには、Colabの Secrets に `GITHUB_TOKEN` を登録してください（または入力欄から認証）。`A100` GPUを選んでください。

実行するのは新しい**実験用ランチャー**です。GPU/Colab本番での学習完了・評価改善は未検証です。チェックポイントは別フォルダーへ保存し元40500を上書きしません。


In [ ]:
from google.colab import drive, userdata
from getpass import getpass
from pathlib import Path
import os, subprocess, sys, tempfile

drive.mount("/content/drive")
STUDY=Path("/content/drive/MyDrive/GeoCPU_v47_fullmodel_20261106")
ARCHIVE=STUDY/"GeoCPU_practical_training_logging_20261107"/"GeoCPU_practical_training_logging_20261107_10k.zip"
OUT=STUDY/"v48_additive_balanced_from40500_20261108"

try:
    TOKEN=userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN=getpass("GitHub token (private repo Content read): ")
if not TOKEN:
    raise RuntimeError("この非公開GitHubをColabから取得するためのトークンが必要です")

ROOT=Path(tempfile.mkdtemp(prefix="geocpu_v48_checkout_",dir="/content"))/"repo"
ASKPASS=ROOT.parent/"git_askpass.sh"
ASKPASS.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo x-access-token ;;\n  *Password*) printf %s "$GH_ACCESS_TOKEN" ;;\nesac\n')
ASKPASS.chmod(0o700)
env=dict(os.environ,GH_ACCESS_TOKEN=TOKEN,GIT_ASKPASS=str(ASKPASS),GIT_TERMINAL_PROMPT="0")
try:
    subprocess.run(
        ["git","-c","credential.helper=","clone","--depth","1",
         "--branch","research/v48-balanced-japanese-curriculum-20261108",
         "https://github.com/corcondor/geocpu-diagnostics.git",str(ROOT)],
        check=True,env=env)
finally:
    ASKPASS.unlink(missing_ok=True)
    TOKEN=""
    env.pop("GH_ACCESS_TOKEN",None)

LAUNCHER=ROOT/"research/v48_balanced_augmentation_20261108/train_from_40500.py"
cmd=[
    sys.executable,"-u",str(LAUNCHER),
    "--study",str(STUDY),
    "--v47-zip",str(ARCHIVE),
    "--output",str(OUT),
    "--minimum-updates","10000",
    "--maximum-updates","20000",
]
print("まずCPU事前検査を行います。元40,500モデルは変更しません。",flush=True)
subprocess.run(cmd,check=True)
print("CPU事前検査は通過しました。A100で学習を開始します。",flush=True)
subprocess.run(cmd+["--execute"],check=True)
